# SAGE-CP v3 — extension studies for the IEEE Access revision

This notebook adds four studies to the v2.1 analysis. The v2.1 results are not changed.

| Study | What it answers | Units of inference |
|---|---|---|
| **R5** | The paper's five held-out regions, now with a **tuned Gibbs–Cherian–Candès (2025) conditional-calibration baseline** | 5 regions (descriptive) |
| **LOSO** | Leave-one-**state**-out on the same Kaggle extract | about 54 states |
| **RAW** | Leave-one-state-out on the **raw CDC BRFSS 2022** file: all respondents (no complete-case filter), metrics also weighted by the design weight `_LLCPWT` | about 54 states |
| **TEMP** | **Temporal + geographic**: models and calibration on raw 2022, evaluation on raw 2023 (design-weighted), by state | 2023 states |
| **LOSO_FULL** | Sensitivity: LOSO with the full v2.1 feature set (LightGBM, one seed) | about 54 states |

**Primary feature set: no_proxy** (the 13 possible post-diagnosis proxies removed). Base models: logistic regression and LightGBM (v2.1 settings). Seeds: R5 0–1, LOSO 0–2, RAW 0, TEMP 0–1. Sets are also built at α = 0.20 for ClassCond, GroupCond and SAGE-CP.

**Before running**
1. Add the input dataset `kamilpytlak/personal-key-indicators-of-heart-disease`.
2. Settings → **Internet ON** (the notebook downloads `LLCP2022XPT.zip` and `LLCP2023XPT.zip` from cdc.gov). If the download fails, download both zips from the CDC BRFSS annual-data pages yourself and attach them as a Kaggle input dataset; the notebook finds them automatically.
3. Accelerator: none (CPU). Use **Save Version → Save & Run All (Commit)** so it runs in the background (expected 6–9 hours; if Kaggle stops it at 12 hours, attach the output and run again: finished units are skipped).

**Resuming.** Every finished unit is saved in `results_v3/units/`. If the session stops, run the notebook again (with the previous output attached as input, or in the same session): finished units are skipped.

**Output.** `/kaggle/working/SAGE_CP_v3_results.zip`. Download it and send it back.

All settings and decision rules are fixed in Section 1 before any result is seen. Target labels are used only for evaluation and for the TargetOracle reference.

In [ ]:
# ======================= 1. Configuration (SAGE-CP v3 extension studies) =======================
# Four studies, each run unit by unit and saved after every unit, so the notebook can be stopped and RESUMED
# (re-running skips units already in results_v3/units.csv):
#   R5   five held-out regions on the Kaggle extract (the paper's design) + the tuned Gibbs-Cherian-Candes baseline
#   LOSO leave-one-state-out on the Kaggle extract (about 54 held-out units instead of 5)
#   RAW  leave-one-state-out on the raw CDC BRFSS 2022 file: all respondents (no complete-case filter), metrics also
#        weighted by the BRFSS design weight _LLCPWT
#   TEMP temporal + geographic: models and calibration on raw 2022, evaluation on raw 2023 (weighted), per state
# Every decision rule and setting is fixed here, before any result is seen.
import os, glob, json, time, math, hashlib, platform, sys, io, zipfile, urllib.request, warnings, traceback
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

CFG3 = dict(
    VERSION="SAGE-CP v3 extension studies (R5, LOSO, RAW, TEMP)",
    STUDIES=["RAW", "TEMP"],                         # run 2 of 2: R5, LOSO and LOSO_FULL finished in run 1 (version 1)
    # full list: ["R5", "LOSO", "RAW", "TEMP", "LOSO_FULL"]
    ALPHA=0.10,
    OUTCOMES=["HadHeartAttack", "HadStroke", "HadDiabetes"],
    MODELS={"R5": ["LR", "LGBM"], "LOSO": ["LR", "LGBM"], "RAW": ["LR", "LGBM"], "TEMP": ["LR", "LGBM"], "LOSO_FULL": ["LGBM"]},
    SEEDS={"R5": [0, 1], "LOSO": [0, 1, 2], "RAW": [0], "TEMP": [0, 1], "LOSO_FULL": [0]},
    # PRIMARY feature set = no_proxy (the 13 possible post-diagnosis proxies removed; v2.1 features_no_proxy.csv).
    # LOSO_FULL repeats leave-one-state-out with the full v2.1 set (Kaggle only) as a sensitivity analysis.
    FEATURES_NO_PROXY=["Sex", "MentalHealthDays", "PhysicalActivities", "SleepHours", "DeafOrHardOfHearing",
                       "BlindOrVisionDifficulty", "DifficultyConcentrating", "SmokerStatus", "ECigaretteUsage",
                       "RaceEthnicityCategory", "AgeCategory", "HeightInMeters", "WeightInKilograms", "BMI",
                       "AlcoholDrinkers", "CovidPos"],
    EXTRA_ALPHAS=[0.20],                              # informativeness at alpha = 0.20 (ClassCond, GroupCond, SAGE-CP)
    N_CAL=20000, N_SRC_TEST=20000, N_DOM=20000, MAX_TRAIN=60000,   # as in v2.1
    CLIP_Q=0.99, DOMAIN_C=0.1, MIN_ESS=1,
    MIN_GROUP=100, MIN_POS=30,                        # eligibility of a group for worst-group metrics (as in v2.1)
    MIN_STATE_ROWS=500,                               # a state with fewer rows is not a held-out unit (rows stay in the source)
    REF_SIMS=2000,                                    # simulations of the exact worst-group reference (Proposition 3)
    # ---- tuned Gibbs, Cherian & Candes (2025) conditional-calibration baseline ----
    GIBBS=dict(studies=["R5", "TEMP"],                # per-unit cost is high; LOSO/RAW/LOSO_FULL skip it
               score_grid=1e-3,                       # scores rounded UP to this grid (conservative) to compress the LP
               n_cal_max=3000,                        # calibration points per label used by the LP (random subsample)
               wbins=[0, 4],                          # candidate numbers of density-ratio bins in the function class
               race=[False, True],                    # candidate: include race indicators
               # selection rule (label-free for the target): for each candidate class, fit the unaugmented quantile
               # regression (level (1-alpha)(n+1)/n), build sets on s_test and estimate every eligible calibrated-group
               # positive-class coverage with the unclipped importance weights; pick the class with the smallest estimated
               # worst-group shortfall below 1-alpha, ties (within tie_tol) broken by the smaller estimated set size.
               # The selected class is then refitted with the exact augmented procedure for the evaluation data.
               tie_tol=0.005),
    BOOT_STATES=5000, BOOT_SEED=2026,                 # bootstrap over held-out units (LOSO, RAW, TEMP)
    TIME_BUDGET_MIN=660,                              # stop starting new units after this; re-run the notebook to resume
    RAW_URLS={2022: "https://www.cdc.gov/brfss/annual_data/2022/files/LLCP2022XPT.zip",
              2023: "https://www.cdc.gov/brfss/annual_data/2023/files/LLCP2023XPT.zip"},
    QUICK=os.environ.get("QUICK", "0") == "1",
)
if CFG3["QUICK"]:   # offline synthetic test only
    CFG3.update(SEEDS={k: [0] for k in CFG3["SEEDS"]}, N_CAL=1500, N_SRC_TEST=1500, N_DOM=1500, MAX_TRAIN=3000,
                MIN_GROUP=30, MIN_POS=8, MIN_STATE_ROWS=200, REF_SIMS=200, BOOT_STATES=300)
    CFG3["GIBBS"].update(n_cal_max=800, wbins=[0, 3])
ALPHA = CFG3["ALPHA"]
OUT = "/kaggle/working/results_v3" if os.path.isdir("/kaggle/working") else os.environ.get("SAGE_OUT", "./results_v3")
os.makedirs(OUT, exist_ok=True)
T0 = time.time()
# resume across sessions: copy finished units from a previous run attached as an input (any folder named results_v3)
import shutil
for f in glob.glob("/kaggle/input/**/results_v3/units/*.csv", recursive=True) + glob.glob("/kaggle/input/**/results_v3/done_units.json", recursive=True):
    dst = f"{OUT}/units/{os.path.basename(f)}" if "/units/" in f else f"{OUT}/{os.path.basename(f)}"
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst): shutil.copy(f, dst)
CFG3_HASH = hashlib.sha256(json.dumps(CFG3, sort_keys=True, default=str).encode()).hexdigest()[:12]
json.dump(dict(CFG3=CFG3, cfg_hash=CFG3_HASH, python=sys.version, platform=platform.platform()),
          open(f"{OUT}/config_v3.json", "w"), indent=2, default=str)
CHECKS = []
def check(name, ok, detail=""):
    CHECKS.append(dict(check=name, passed=bool(ok), detail=str(detail)))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + ("" if ok else f" | {detail}"))
    return bool(ok)
print(CFG3["VERSION"], "| config hash", CFG3_HASH, "| output", OUT)


In [ ]:
# ======================= 2. Data: Kaggle extract and raw CDC BRFSS files, mapped to one standard frame =======================
# Standard frame: state (name), Sex, Age (18-44 / 45-64 / 65+), Race, the three outcomes coded 0/1/NaN, wdesign (BRFSS
# design weight; 1 for the Kaggle extract) and the model features. Outcomes and other diagnoses never enter a feature set.
FIPS = {1: "Alabama", 2: "Alaska", 4: "Arizona", 5: "Arkansas", 6: "California", 8: "Colorado", 9: "Connecticut", 10: "Delaware",
        11: "District of Columbia", 12: "Florida", 13: "Georgia", 15: "Hawaii", 16: "Idaho", 17: "Illinois", 18: "Indiana", 19: "Iowa",
        20: "Kansas", 21: "Kentucky", 22: "Louisiana", 23: "Maine", 24: "Maryland", 25: "Massachusetts", 26: "Michigan", 27: "Minnesota",
        28: "Mississippi", 29: "Missouri", 30: "Montana", 31: "Nebraska", 32: "Nevada", 33: "New Hampshire", 34: "New Jersey",
        35: "New Mexico", 36: "New York", 37: "North Carolina", 38: "North Dakota", 39: "Ohio", 40: "Oklahoma", 41: "Oregon",
        42: "Pennsylvania", 44: "Rhode Island", 45: "South Carolina", 46: "South Dakota", 47: "Tennessee", 48: "Texas", 49: "Utah",
        50: "Vermont", 51: "Virginia", 53: "Washington", 54: "West Virginia", 55: "Wisconsin", 56: "Wyoming", 66: "Guam",
        72: "Puerto Rico", 78: "Virgin Islands"}
REGION = {}
for s in ["Connecticut", "Maine", "Massachusetts", "New Hampshire", "Rhode Island", "Vermont", "New Jersey", "New York", "Pennsylvania"]: REGION[s] = "Northeast"
for s in ["Illinois", "Indiana", "Michigan", "Ohio", "Wisconsin", "Iowa", "Kansas", "Minnesota", "Missouri", "Nebraska", "North Dakota", "South Dakota"]: REGION[s] = "Midwest"
for s in ["Delaware", "District of Columbia", "Florida", "Georgia", "Maryland", "North Carolina", "South Carolina", "Virginia", "West Virginia",
          "Alabama", "Kentucky", "Mississippi", "Tennessee", "Arkansas", "Louisiana", "Oklahoma", "Texas"]: REGION[s] = "South"
for s in ["Arizona", "Colorado", "Idaho", "Montana", "Nevada", "New Mexico", "Utah", "Wyoming", "Alaska", "California", "Hawaii", "Oregon", "Washington"]: REGION[s] = "West"
for s in ["Guam", "Puerto Rico", "Virgin Islands"]: REGION[s] = "Territories"
REGIONS_ALL = ["Northeast", "Midwest", "South", "West", "Territories"]

def age3(lower):
    return np.where(lower < 45, "18-44", np.where(lower < 65, "45-64", "65+"))

def _find(name):
    hits = []
    for p in [f"/kaggle/input/**/{name}", f"./{name}", f"./data/{name}"]:
        hits += [h for h in sorted(glob.glob(p, recursive=True)) if h not in hits]
    return hits

def load_kaggle():
    """heart_2022_no_nans.csv exactly as used by v2.1 (complete cases, 246,022 rows)."""
    hits = _find("heart_2022_no_nans.csv")
    if not hits: raise FileNotFoundError("Add the Kaggle dataset 'kamilpytlak/personal-key-indicators-of-heart-disease' as notebook input.")
    d = pd.read_csv(hits[0]); check("Kaggle file has 246,022 rows x 40 columns", d.shape == (246022, 40) or CFG3["QUICK"], d.shape)
    F = pd.DataFrame({"state": d["State"].astype(str), "Sex": d["Sex"].astype(str)})
    F["Age"] = age3(d["AgeCategory"].astype(str).str.extract(r"(\d+)")[0].astype(float).to_numpy())
    F["Race"] = d["RaceEthnicityCategory"].astype(str)
    for o in CFG3["OUTCOMES"]:
        v = d[o].astype(str).str.strip().str.lower()
        F[o] = np.where(v.str.startswith("yes"), 1.0, np.where(v.str.startswith("no"), 0.0, np.nan))
        if o == "HadDiabetes":   # yes_only coding (v2.1 primary): 'Yes' = 1; 'No', borderline and pregnancy-only = 0
            F[o] = np.where(v == "yes", 1.0, np.where(v.isin(["no", "no, pre-diabetes or borderline diabetes",
                                                               "yes, but only during pregnancy (female)"]), 0.0, np.nan))
    F["wdesign"] = 1.0
    other = [c for c in d.columns if c.startswith("Had")]
    feats = [c for c in d.columns if c not in other + ["State"]]
    num = [c for c in feats if pd.api.types.is_numeric_dtype(d[c])]; cat = [c for c in feats if c not in num]
    X = d[feats].copy()
    for c in cat: X[c] = X[c].astype(str)
    return F.reset_index(drop=True), X.reset_index(drop=True), num, cat, dict(source="Kaggle heart_2022_no_nans.csv", file=hits[0], n=len(d))

# ---- raw BRFSS: Kaggle-equivalent features from the CDC codebook variables ----
# Each feature lists candidate variable names (first present is used, so 2022 and 2023 names can differ); categorical
# features use the raw response code, with the listed codes treated as missing. Features that are missing in a year
# are dropped and logged. ChestScan is not used (not in the core questionnaire; a post-diagnostic marker per [28]).
RAW_FEATURES = {
    # name: (candidates, kind, missing codes or mapper)
    "Sex": (["SEXVAR", "_SEX"], "cat", []),
    "AgeCategory": (["_AGEG5YR"], "cat", [14]),
    "RaceEthnicityCategory": (["_IMPRACE", "_RACEPR1"], "cat", []),
    "GeneralHealth": (["GENHLTH"], "cat", [7, 9]),
    "PhysicalHealthDays": (["PHYSHLTH"], "days", None),
    "MentalHealthDays": (["MENTHLTH"], "days", None),
    "LastCheckupTime": (["CHECKUP1"], "cat", [7, 9]),
    "PhysicalActivities": (["EXERANY2"], "cat", [7, 9]),
    "SleepHours": (["SLEPTIM1"], "sleep", None),
    "RemovedTeeth": (["RMVTETH4"], "cat", [7, 9]),
    "DeafOrHardOfHearing": (["DEAF"], "cat", [7, 9]),
    "BlindOrVisionDifficulty": (["BLIND"], "cat", [7, 9]),
    "DifficultyConcentrating": (["DECIDE"], "cat", [7, 9]),
    "DifficultyWalking": (["DIFFWALK"], "cat", [7, 9]),
    "DifficultyDressingBathing": (["DIFFDRES"], "cat", [7, 9]),
    "DifficultyErrands": (["DIFFALON"], "cat", [7, 9]),
    "SmokerStatus": (["_SMOKER3"], "cat", [9]),
    "ECigaretteUsage": (["ECIGNOW2", "ECIGNOW3", "ECIGNOW1"], "cat", [7, 9]),
    "HeightInMeters": (["HTM4"], "scale100", None),
    "WeightInKilograms": (["WTKG3"], "scale100", None),
    "BMI": (["_BMI5"], "scale100", None),
    "AlcoholDrinkers": (["DRNKANY6", "DRNKANY5"], "cat", [7, 9]),
    "HIVTesting": (["HIVTST7"], "cat", [7, 9]),
    "FluVaxLast12": (["FLUSHOT7"], "cat", [7, 9]),
    "PneumoVaxEver": (["PNEUVAC4"], "cat", [7, 9]),
    "TetanusLast10Tdap": (["TETANUS1"], "cat", [7, 9]),
    "HighRiskLastYear": (["HIVRISK5"], "cat", [7, 9]),
    "CovidPos": (["COVIDPOS", "COVIDPO1"], "cat", [7, 9]),
}
RAW_OUTCOMES = {"HadHeartAttack": ["CVDINFR4"], "HadStroke": ["CVDSTRK3"], "HadDiabetes": ["DIABETE4"]}
RAW_DESIGN = {"wdesign": ["_LLCPWT"], "strata": ["_STSTR"], "psu": ["_PSU"]}

def _raw_file(year):
    """Local copy if attached as a Kaggle input (LLCP{year}.XPT or the zip); otherwise download from CDC (Internet ON)."""
    for nm in [f"LLCP_{year}.XPT", f"LLCP{year}.XPT", f"LLCP{year}.XPT ", f"LLCP{year}XPT.zip", f"llcp{year}.xpt", f"LLCP_{year}.xpt"]:
        h = _find(nm)
        if h: return h[0]
    dest = f"/kaggle/working/LLCP{year}XPT.zip" if os.path.isdir("/kaggle/working") else f"./LLCP{year}XPT.zip"
    if not os.path.exists(dest):
        print(f"downloading BRFSS {year} from CDC ...", flush=True)
        try:
            urllib.request.urlretrieve(CFG3["RAW_URLS"][year], dest)
        except Exception as e:
            raise RuntimeError(f"Could not download {CFG3['RAW_URLS'][year]} ({e}). In Kaggle: Settings -> Internet ON, or attach "
                               f"the file LLCP{year}XPT.zip as an input dataset.")
    return dest

def _read_xpt(path, keep):
    if path.lower().endswith(".zip"):
        z = zipfile.ZipFile(path); inner = [n for n in z.namelist() if n.strip().lower().endswith(".xpt")][0]
        path = z.extract(inner, os.path.dirname(path) or "."); path_clean = path.strip()
        if path_clean != path: os.replace(path, path_clean); path = path_clean
    parts = []
    for ch in pd.read_sas(path, format="xport", chunksize=50000):
        ch.columns = [c.strip().upper() for c in ch.columns]
        parts.append(ch[[c for c in keep if c in ch.columns]])
    return pd.concat(parts, ignore_index=True)

RAW_MAP_LOG = []
def load_raw(year, frame=None):
    """frame: optional pre-read DataFrame with raw CDC variable names (used by the offline test)."""
    keep = sorted({c for v in RAW_FEATURES.values() for c in v[0]} | {c for v in RAW_OUTCOMES.values() for c in v} |
                  {c for v in RAW_DESIGN.values() for c in v} | {"_STATE"})
    d = frame if frame is not None else _read_xpt(_raw_file(year), keep)
    n0 = len(d)
    def pick(cands):
        for c in cands:
            if c in d.columns: return c
        return None
    F = pd.DataFrame({"state": d["_STATE"].round().astype(int).map(FIPS)})
    agec = d["_AGEG5YR"].to_numpy(float); lower = np.where((agec >= 1) & (agec <= 13), 18 + np.maximum(0, agec - 1) * 5 + (agec > 1) * 2, np.nan)
    lower = np.where(agec == 1, 18, lower)                     # 1: 18-24, 2: 25-29, ..., 13: 80+
    F["Sex"] = np.where(d[pick(["SEXVAR", "_SEX"])] == 1, "Male", np.where(d[pick(["SEXVAR", "_SEX"])] == 2, "Female", "__MISSING__"))
    F["Age"] = np.where(np.isnan(lower), "__MISSING__", age3(np.nan_to_num(lower, nan=0)))
    rc = d[pick(["_IMPRACE", "_RACEPR1"])].to_numpy(float)
    F["Race"] = np.select([rc == 1, rc == 2, rc == 5, np.isin(rc, [3, 4, 6, 7])],
                          ["White, Non-Hispanic", "Black, Non-Hispanic", "Hispanic", "Other or multiracial, Non-Hispanic"], "__MISSING__")
    for o, cands in RAW_OUTCOMES.items():
        v = d[pick(cands)].to_numpy(float)
        F[o] = np.where(v == 1, 1.0, np.where(v == 2, 0.0, np.nan)) if o != "HadDiabetes" else \
               np.where(v == 1, 1.0, np.where(np.isin(v, [2, 3, 4]), 0.0, np.nan))   # yes_only, as v2.1
        RAW_MAP_LOG.append(dict(year=year, item=o, variable=pick(cands), kind="outcome"))
    for k, cands in RAW_DESIGN.items():
        c = pick(cands); F[k] = d[c].to_numpy(float) if c else np.nan
        RAW_MAP_LOG.append(dict(year=year, item=k, variable=c, kind="design"))
    X = pd.DataFrame(index=d.index); num, cat = [], []
    for name, (cands, kind, miss) in RAW_FEATURES.items():
        c = pick(cands)
        RAW_MAP_LOG.append(dict(year=year, item=name, variable=c, kind=kind))
        if c is None: continue
        v = d[c].to_numpy(float)
        if kind == "cat":
            X[name] = np.where(np.isnan(v) | np.isin(v, miss), "__MISSING__", pd.Series(v).round().astype("Int64").astype(str).to_numpy()); cat.append(name)
        elif kind == "days":
            X[name] = np.where(v == 88, 0.0, np.where((v >= 1) & (v <= 30), v, np.nan)); num.append(name)
        elif kind == "sleep":
            X[name] = np.where((v >= 1) & (v <= 24), v, np.nan); num.append(name)
        elif kind == "scale100":
            X[name] = np.where(v > 0, v / 100.0, np.nan); num.append(name)
    ok = F["state"].notna() & (F["Sex"] != "__MISSING__") & (F["Age"] != "__MISSING__") & (F["Race"] != "__MISSING__") & \
         F[CFG3["OUTCOMES"]].notna().any(axis=1) & (F["wdesign"] > 0)
    info = dict(source=f"CDC BRFSS LLCP{year}", n_raw=n0, n_kept=int(ok.sum()),
                n_dropped_missing_group_or_state=int((~ok).sum()),
                features_used=num + cat, features_missing=[n for n, (cands, _, _) in RAW_FEATURES.items() if pick(cands) is None])
    print(f"BRFSS {year}: {n0} raw rows, {int(ok.sum())} kept; features missing this year: {info['features_missing']}")
    return F[ok].reset_index(drop=True), X[ok].reset_index(drop=True), num, cat, info


In [ ]:
# ======================= 3. Models, conformal methods, metrics and the exact worst-group reference =======================
# weighted_q, conformal_sets, sage_sets, kish, _qr_lp, fit_domain, weights_from and the reference are VERBATIM from v2.1.
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from scipy.optimize import linprog
from scipy.sparse import csr_matrix, hstack as sp_hstack, identity as sp_identity
from scipy.stats import wilcoxon, binomtest

MODEL_BACKEND = {}
def make_model(name, seed):
    """Same settings as v2.1 (Table S4)."""
    if name == "LR":
        MODEL_BACKEND[name] = "sklearn LogisticRegression"
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, C=1.0))
    try:
        import lightgbm as lgb; MODEL_BACKEND[name] = f"lightgbm {lgb.__version__}"
        return lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.8, random_state=seed, verbose=-1)
    except ImportError:
        MODEL_BACKEND[name] = "FALLBACK sklearn HistGradientBoosting"
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=seed)

def make_encoder(num, cat):
    return ColumnTransformer([("num", SimpleImputer(strategy="median"), num),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), cat)],
                             remainder="drop", verbose_feature_names_out=False)

def weighted_q(cal_scores, cal_w, test_w, alpha):
    o = np.argsort(cal_scores); s = cal_scores[o]; cw = np.cumsum(cal_w[o]); W = cw[-1]
    k = np.searchsorted(cw, (1 - alpha) * (W + test_w), side="left")
    return np.where(k < len(s), s[np.minimum(k, len(s) - 1)], np.inf)

def conformal_sets(p_cal, y_cal, p_te, alpha, w_cal=None, w_te=None, class_cond=False):
    n_te = len(p_te)
    w_cal = np.ones(len(p_cal)) if w_cal is None else w_cal
    w_te = np.ones(n_te) if w_te is None else w_te
    P_te, P_cal = np.c_[1 - p_te, p_te], np.c_[1 - p_cal, p_cal]
    S = np.zeros((n_te, 2), bool)
    if class_cond:
        for c in (0, 1):
            m = y_cal == c
            S[:, c] = (1 - P_te[:, c]) <= (weighted_q(1 - P_cal[m, c], w_cal[m], w_te, alpha) if m.any() else np.inf)
    else:
        q = weighted_q(1 - P_cal[np.arange(len(y_cal)), y_cal], w_cal, w_te, alpha)
        S = (1 - P_te) <= q[:, None]
    return S

def sage_sets(p_cal, y_cal, keys_cal, p_te, keys_te, alpha, w_cal=None, w_te=None, min_ess=100):
    n_te = len(p_te)
    w_cal = np.ones(len(p_cal)) if w_cal is None else w_cal
    w_te = np.ones(n_te) if w_te is None else w_te
    P_te, P_cal = np.c_[1 - p_te, p_te], np.c_[1 - p_cal, p_cal]
    S = np.zeros((n_te, 2), bool)
    for c in (0, 1):
        mc = y_cal == c; done = np.zeros(n_te, bool)
        for L, (kc, kt) in enumerate(zip(keys_cal, keys_te)):
            last = L == len(keys_cal) - 1
            for k in np.unique(kt[~done]):
                ti = np.where(~done & (kt == k))[0]
                cm = mc & (kc == k); wc = w_cal[cm]
                ess = wc.sum() ** 2 / (wc ** 2).sum() if cm.any() else 0.0
                if ess < min_ess and not last: continue
                q = weighted_q(1 - P_cal[cm, c], wc, w_te[ti], alpha) if cm.any() else np.full(len(ti), np.inf)
                S[ti, c] = (1 - P_te[ti, c]) <= q
                done[ti] = True
    return S

def kish(w):
    w = np.asarray(w, float)
    return float(w.sum() ** 2 / (w ** 2).sum()) if len(w) and (w ** 2).sum() > 0 else 0.0

def _qr_lp(Xd, s, w, tau):
    n, d = Xd.shape
    c = np.r_[np.zeros(d), tau * w, (1 - tau) * w]
    A = sp_hstack([csr_matrix(Xd), sp_identity(n), -sp_identity(n)]).tocsr()
    res = linprog(c, A_eq=A, b_eq=s, bounds=[(None, None)] * d + [(0, None)] * (2 * n), method="highs")
    if not res.success: raise RuntimeError("quantile-regression LP failed: " + res.message)
    return res.x[:d]

def fit_domain(Xs, Xt):
    Xd = np.r_[Xs, Xt]; yd = np.r_[np.zeros(len(Xs)), np.ones(len(Xt))]
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, C=CFG3["DOMAIN_C"])).fit(Xd, yd)
    m.log_prior_ = np.log(len(Xs) / len(Xt))
    return m

def weights_from(m, Xq):
    return np.exp(np.clip(m.decision_function(Xq) + m.log_prior_, -30, 30))

# ---------------- tuned conditional calibration (Gibbs, Cherian & Candes, JRSSB 2025) ----------------
# Function class F = span of indicator features: intercept, Sex x Age cells, optionally race, optionally B bins of the
# estimated density ratio (bins = quantiles on s_dom, label-free). For a finite indicator class the augmented
# quantile-regression threshold g_S(x) depends on x only through its pattern (cell, race, bin). We use the
# deterministic conservative form: the test score is imputed at its maximum (S = 1 for the LAC score), so the set
# {y : S(x, y) <= g_1(x)} contains the exact non-randomised Gibbs set and inherits its coverage guarantee
# (P(Y in C(X) | X in group) >= 1 - alpha for every group in F; weighted versions for shifts in the span of F).
# Fitted per label (label-conditional), one LP per (label, pattern present in the evaluation data).
def _patterns(G, cols):
    return pd.Series(list(map(tuple, G[cols].to_numpy())))

def gibbs_thresholds(s_cal, D_cal, D_pat, tau, grid):
    """s_cal: calibration scores of one label; D_cal: their indicator rows; D_pat: one indicator row per pattern.
    Returns threshold per pattern (augmented fit with the pattern's test point at score 1)."""
    s = np.ceil(s_cal / grid) * grid
    key = pd.DataFrame(D_cal).astype(np.int8).astype(str).agg("".join, axis=1) + "|" + pd.Series(s).round(6).astype(str)
    comp = pd.DataFrame({"k": key, "i": np.arange(len(s))}).groupby("k")["i"].agg(["first", "size"])
    Dc, sc, wc = D_cal[comp["first"].to_numpy()], s[comp["first"].to_numpy()], comp["size"].to_numpy(float)
    act = D_cal.sum(0) > 0
    th = np.empty(len(D_pat))
    for j, dp in enumerate(D_pat):
        a = act | (dp > 0)
        beta = _qr_lp(np.r_[Dc, dp[None, :]][:, a], np.r_[sc, 1.0], np.r_[wc, 1.0], tau)
        th[j] = float(dp[a] @ beta)
    return th

def gibbs_sets(p_cal, y_cal, Gc, p_ev_list, G_ev_list, cols, alpha, grid):
    """cols: indicator columns of the function class (Gc / G_ev: DataFrames of 0/1 indicators incl. 'one')."""
    outs = [np.zeros((len(p), 2), bool) for p in p_ev_list]
    pats = pd.concat([G[cols] for G in G_ev_list]).drop_duplicates().to_numpy(float)
    pidx = {tuple(r): j for j, r in enumerate(pats)}
    for c in (0, 1):
        m = y_cal == c; n = int(m.sum())
        if n == 0:
            for S in outs: S[:, c] = True
            continue
        tau = 1 - alpha                                    # augmented fit: no finite-sample correction needed
        sc = 1 - (p_cal[m] if c == 1 else 1 - p_cal[m])
        th = gibbs_thresholds(sc, Gc[cols].to_numpy(float)[m], pats, tau, grid)
        for S, p, G in zip(outs, p_ev_list, G_ev_list):
            j = np.array([pidx[tuple(r)] for r in G[cols].to_numpy(float)])
            sc_ev = 1 - (p if c == 1 else 1 - p)
            S[:, c] = sc_ev <= th[j] + 1e-9
    return outs

# ---------------- metrics (optionally weighted by the BRFSS design weight) ----------------
def _wmean(x, w):
    return float(np.sum(w * x) / np.sum(w)) if len(x) and np.sum(w) > 0 else np.nan

def metrics(S, y, G, w=None, eligible=None):
    """Coverage, positive-class coverage, set size, share of {0,1}, empty and singleton-{1} rates, and the minimum
    positive-class coverage over ELIGIBLE groups (Sex, Age, SexxAge: 'cal'; Race: 'race'). Eligibility (MIN_GROUP rows and
    MIN_POS positives in the evaluation set) is decided on counts, so weighted and unweighted metrics use the same groups."""
    w = np.ones(len(y)) if w is None else w
    cov = S[np.arange(len(y)), y].astype(float); size = S.sum(1); pos = y == 1
    out = dict(n=len(y), n_pos=int(pos.sum()), coverage=_wmean(cov, w), cov_pos=_wmean(cov[pos], w[pos]),
               cov_neg=_wmean(cov[~pos], w[~pos]), set_size=_wmean(size, w), both=_wmean((size == 2).astype(float), w),
               empty=_wmean((size == 0).astype(float), w), single1=_wmean((S[:, 1] & ~S[:, 0]).astype(float), w),
               ppv_single1=_wmean(y[S[:, 1] & ~S[:, 0]].astype(float), w[S[:, 1] & ~S[:, 0]]))
    out["absdev_cov_pos"] = abs(out["cov_pos"] - (1 - ALPHA)) if not np.isnan(out["cov_pos"]) else np.nan
    for name, axes in [("cal", ["Sex", "Age", "SexxAge"]), ("race", ["Race"])]:
        vals = []
        for ax in axes:
            for g, ix in G.groupby(ax).indices.items():
                if len(ix) >= CFG3["MIN_GROUP"] and int(y[ix].sum()) >= CFG3["MIN_POS"]:
                    pi = ix[y[ix] == 1]; vals.append(_wmean(cov[pi], w[pi]))
        out[f"wg_{name}_pos"] = float(np.min(vals)) if vals else np.nan
        out[f"n_elig_{name}"] = len(vals)
    return out

def _beta_pi(ncell, alpha, sims, rng):
    n = int(ncell)
    if n <= 0: return np.ones(sims)
    k = math.ceil((1 - alpha) * (n + 1))
    return np.ones(sims) if k > n else rng.beta(k, n + 1 - k, size=sims)

def exact_reference(y_ev, G_ev, cal_n, alpha, sims, rng):
    """Proposition 3: law of the minimum positive-class coverage over eligible calibrated groups under label x cell split
    CP with cal_n[cell] calibration positives (exact for continuous scores; cells without positives use the Age parent).
    Returns mean and 5th percentile."""
    pos = np.where(y_ev == 1)[0]
    if not len(pos): return np.nan, np.nan
    cell = G_ev["SexxAge"].to_numpy(str)[pos]
    def n_eff(c):
        v = cal_n.get(c, 0)
        return v if v > 0 else sum(x for k, x in cal_n.items() if k.split("|")[-1] == c.split("|")[-1])
    cells = sorted(set(cell)); ci = np.searchsorted(np.array(cells), cell)
    pi = np.column_stack([_beta_pi(n_eff(c), alpha, sims, rng) for c in cells])[:, ci]
    M = []
    for ax in ["Sex", "Age", "SexxAge"]:
        for g, ix in G_ev.groupby(ax).indices.items():
            if len(ix) >= CFG3["MIN_GROUP"] and int(y_ev[ix].sum()) >= CFG3["MIN_POS"]:
                v = np.zeros(len(y_ev), bool); v[ix] = True; M.append(v[pos])
    if not M: return np.nan, np.nan
    M = np.array(M, np.float32).T
    Z = (rng.random((sims, len(pos))) < pi).astype(np.float32)
    mn = ((Z @ M) / M.sum(0)).min(1)
    return float(mn.mean()), float(np.quantile(mn, 0.05))

def gibbs_unaug_sets(p_cal, y_cal, Gc, p_ev, G_ev, cols, alpha, grid):
    """Unaugmented (split) fit used ONLY to select the function class on source data: one LP per label at level
    (1-alpha)(n+1)/n; threshold = D(x) beta."""
    S = np.zeros((len(p_ev), 2), bool)
    for c in (0, 1):
        m = y_cal == c; n = int(m.sum())
        tau = 1.0 if n == 0 else min(1.0, (1 - alpha) * (n + 1) / n)
        if tau >= 1.0: S[:, c] = True; continue
        sc = np.ceil((1 - (p_cal[m] if c == 1 else 1 - p_cal[m])) / grid) * grid
        D = Gc[cols].to_numpy(float)[m]
        key = pd.DataFrame(D).astype(np.int8).astype(str).agg("".join, axis=1) + "|" + pd.Series(sc).round(6).astype(str)
        comp = pd.DataFrame({"k": key, "i": np.arange(len(sc))}).groupby("k")["i"].agg(["first", "size"])
        act = D.sum(0) > 0; beta = np.zeros(D.shape[1])
        beta[act] = _qr_lp(D[comp["first"].to_numpy()][:, act], sc[comp["first"].to_numpy()], comp["size"].to_numpy(float), tau)
        S[:, c] = (1 - (p_ev if c == 1 else 1 - p_ev)) <= G_ev[cols].to_numpy(float) @ beta + 1e-9
    return S


In [ ]:
# ======================= 4. Engine: one held-out unit (disjoint partitions, train-only preprocessing) =======================
# Source: s_cal | s_test | s_dom | s_train (disjoint, as v2.1). Target: t_unlab (domain classifier; labels used ONLY by the
# TargetOracle reference) | t_eval (evaluation only). Encoders: base model on s_train covariates; domain model on
# s_dom + t_unlab covariates. No preprocessing step reads an outcome. Weights: split mode, clipped at the CLIP_Q quantile
# of the s_dom weights (v2.1 primary). Gibbs class selection uses source data and unlabelled target covariates only.
UNIT_DIR, DONE_JSON = f"{OUT}/units", f"{OUT}/done_units.json"; os.makedirs(UNIT_DIR, exist_ok=True)
UNITS_CSV, GIBBS_LOG = f"{OUT}/units.csv", f"{OUT}/gibbs_selection.csv"   # assembled from UNIT_DIR in Section 6
DONE = set(map(tuple, json.load(open(DONE_JSON)))) if os.path.exists(DONE_JSON) else set()

def _save(rows, path):   # one file per unit (column sets differ between studies; files are concatenated in Section 6)
    if rows: pd.DataFrame(rows).to_csv(path, index=False)

def _cap(idx, n, rng): return idx if len(idx) <= n else rng.choice(idx, n, replace=False)

def gframe(F, rows):
    G = F.iloc[rows][["Sex", "Age", "Race"]].reset_index(drop=True).astype(str)
    G["SexxAge"] = G["Sex"] + "|" + G["Age"]; return G

SA_CATS = [f"{s}|{a}" for s in ["Female", "Male"] for a in ["18-44", "45-64", "65+"]]
def indicators(G, wraw, race_cats, edges):
    D = pd.DataFrame({f"SA={c}": (G["SexxAge"] == c).astype(float).to_numpy() for c in SA_CATS})
    for r in race_cats[1:]: D[f"R={r}"] = (G["Race"] == r).astype(float).to_numpy()
    if edges is not None:
        b = np.digitize(wraw, edges)
        for k in range(1, len(edges) + 1): D[f"W={k}"] = (b == k).astype(float)
    return D

def run_unit(study, unit, seed, Fs, Xs, src, Ft, Xt, tgt, num, cat, eval_by=None, unit_index=0, feat_tag="no_proxy"):
    """Fs/Xs, Ft/Xt: standard frames and features of source and target (same objects for within-year designs).
    src / tgt: row indices. eval_by: optional column of Ft by which t_eval is split into evaluation units (TEMP: state)."""
    key = (study, str(unit), int(seed))
    if key in DONE: return "done"
    t0 = time.time(); rng = np.random.default_rng(seed * 1_000_003 + unit_index * 7919 + ["R5", "LOSO", "RAW", "TEMP", "LOSO_FULL"].index(study))
    src = rng.permutation(src); tgt = rng.permutation(tgt)
    n_cal, n_st, n_dom = (min(CFG3[k], len(src) // 5) for k in ["N_CAL", "N_SRC_TEST", "N_DOM"])
    a = 0; P = {}
    P["s_cal"] = src[a:a + n_cal]; a += n_cal; P["s_test"] = src[a:a + n_st]; a += n_st; P["s_dom"] = src[a:a + n_dom]; a += n_dom
    P["s_train"] = _cap(src[a:], CFG3["MAX_TRAIN"], rng)
    nu = min(len(tgt) // 2, 20000) if eval_by is not None else len(tgt) // 2
    P["t_unlab"], P["t_eval"] = tgt[:nu], tgt[nu:]
    assert len(np.unique(np.r_[P["s_cal"], P["s_test"], P["s_dom"], P["s_train"]])) == sum(len(P[k]) for k in ["s_cal", "s_test", "s_dom", "s_train"])
    if Ft is Fs: assert not np.isin(P["t_eval"], np.r_[P["s_cal"], P["s_test"], P["s_dom"], P["s_train"], P["t_unlab"]]).any()
    SRC_PARTS, TGT_PARTS = ["s_cal", "s_test", "s_dom", "s_train"], ["t_unlab", "t_eval"]
    Xof = lambda k: (Xs if k in SRC_PARTS else Xt).iloc[P[k]]
    Fof = lambda k: (Fs if k in SRC_PARTS else Ft)
    enc_b = make_encoder(num, cat).fit(Xof("s_train"))
    XB = {k: enc_b.transform(Xof(k)).astype(np.float32) for k in ["s_train", "s_cal", "s_test", "t_unlab", "t_eval"]}
    enc_d = make_encoder(num, cat).fit(pd.concat([Xof("s_dom"), Xof("t_unlab")]))
    XD = {k: enc_d.transform(Xof(k)).astype(np.float32) for k in ["s_cal", "s_test", "s_dom", "t_unlab", "t_eval"]}
    dm = fit_domain(XD["s_dom"], XD["t_unlab"])
    WR = {k: weights_from(dm, XD[k]) for k in XD}
    clipv = float(np.quantile(WR["s_dom"], CFG3["CLIP_Q"])); WC = {k: np.minimum(v, clipv) for k, v in WR.items()}
    G = {k: gframe(Fof(k), P[k]) for k in ["s_cal", "s_test", "t_unlab", "t_eval"]}
    race_cats = sorted(pd.unique(pd.concat([G["s_cal"]["Race"], G["t_eval"]["Race"]])))
    rows, glog = [], []
    for oi, o in enumerate(CFG3["OUTCOMES"]):
        yv = {k: Fof(k)[o].to_numpy()[P[k]] for k in P}; ok = {k: ~np.isnan(yv[k]) for k in P}
        y = {k: yv[k][ok[k]].astype(int) for k in P}
        if y["t_eval"].sum() < 1 or y["s_cal"].sum() < 1: continue
        for mi, mname in enumerate(CFG3["MODELS"][study]):
            model = make_model(mname, seed).fit(XB["s_train"][ok["s_train"]], y["s_train"])
            pr = {k: model.predict_proba(XB[k][ok[k]])[:, 1] for k in ["s_cal", "s_test", "t_unlab", "t_eval"]}
            g = {k: G[k][ok[k]].reset_index(drop=True) for k in G}
            w = {k: (WC[k][ok[k]], WR[k][ok[k]]) for k in ["s_cal", "s_test", "t_unlab", "t_eval"]}
            keys = lambda k: [g[k]["SexxAge"].to_numpy(str), g[k]["Age"].to_numpy(str), np.array(["all"] * len(g[k]))]
            pc, yc = pr["s_cal"], y["s_cal"]; wc, wte = w["s_cal"][0], w["t_eval"][0]
            SETS = {
                "ClassCond": conformal_sets(pc, yc, pr["t_eval"], ALPHA, class_cond=True),
                "GroupCond": sage_sets(pc, yc, keys("s_cal"), pr["t_eval"], keys("t_eval"), ALPHA, min_ess=CFG3["MIN_ESS"]),
                "Weighted+ClassCond": conformal_sets(pc, yc, pr["t_eval"], ALPHA, w_cal=wc, w_te=wte, class_cond=True),
                "SAGE-CP": sage_sets(pc, yc, keys("s_cal"), pr["t_eval"], keys("t_eval"), ALPHA, w_cal=wc, w_te=wte, min_ess=CFG3["MIN_ESS"]),
                "TargetOracle": sage_sets(pr["t_unlab"], y["t_unlab"], keys("t_unlab"), pr["t_eval"], keys("t_eval"), ALPHA, min_ess=CFG3["MIN_ESS"]),
            }
            for a2 in CFG3["EXTRA_ALPHAS"]:   # informativeness at a looser level (coverage targets 1 - a2)
                SETS[f"ClassCond@{a2:.2f}"] = conformal_sets(pc, yc, pr["t_eval"], a2, class_cond=True)
                SETS[f"GroupCond@{a2:.2f}"] = sage_sets(pc, yc, keys("s_cal"), pr["t_eval"], keys("t_eval"), a2, min_ess=CFG3["MIN_ESS"])
                SETS[f"SAGE-CP@{a2:.2f}"] = sage_sets(pc, yc, keys("s_cal"), pr["t_eval"], keys("t_eval"), a2, w_cal=wc, w_te=wte, min_ess=CFG3["MIN_ESS"])
            gib_info = {}
            if study in CFG3["GIBBS"]["studies"]:
                tg = time.time(); GB = CFG3["GIBBS"]
                sub = np.r_[_cap(np.where(yc == 0)[0], GB["n_cal_max"], rng), _cap(np.where(yc == 1)[0], GB["n_cal_max"], rng)]
                cand = []
                for B in GB["wbins"]:
                    edges = None if B == 0 else np.quantile(WR["s_dom"], np.linspace(0, 1, B + 1)[1:-1])
                    for use_race in GB["race"]:
                        rc = race_cats if use_race else race_cats[:1]
                        Dc = indicators(g["s_cal"].iloc[sub].reset_index(drop=True), w["s_cal"][1][sub], rc, edges)
                        Ds = indicators(g["s_test"], w["s_test"][1], rc, edges)
                        cols = list(Dc.columns)
                        S_st = gibbs_unaug_sets(pc[sub], yc[sub], Dc, pr["s_test"], Ds, cols, ALPHA, GB["score_grid"])
                        est = metrics(S_st, y["s_test"], g["s_test"], w=w["s_test"][1])     # importance-weighted estimate of target
                        short = max(0.0, (1 - ALPHA) - est["wg_cal_pos"]) if not np.isnan(est["wg_cal_pos"]) else np.inf
                        cand.append(dict(B=B, race=use_race, edges=edges, cols=cols, short=short, size=est["set_size"]))
                        glog.append(dict(study=study, unit=unit, seed=seed, outcome=o, model=mname, wbins=B, race=use_race, n_cols=len(cols),
                                         est_wg_cal_pos=est["wg_cal_pos"], est_shortfall=short, est_set_size=est["set_size"]))
                best = min(c["short"] for c in cand)
                sel = min([c for c in cand if c["short"] <= best + GB["tie_tol"]], key=lambda c: c["size"])
                rc = race_cats if sel["race"] else race_cats[:1]
                Dc = indicators(g["s_cal"].iloc[sub].reset_index(drop=True), w["s_cal"][1][sub], rc, sel["edges"])
                De = indicators(g["t_eval"], w["t_eval"][1], rc, sel["edges"])
                SETS["Gibbs-tuned"] = gibbs_sets(pc[sub], yc[sub], Dc, [pr["t_eval"]], [De], sel["cols"], ALPHA, GB["score_grid"])[0]
                gib_info = dict(gibbs_wbins=sel["B"], gibbs_race=sel["race"], gibbs_seconds=time.time() - tg)
                for r in glog[-len(cand):]: r["selected"] = (r["wbins"] == sel["B"]) and (r["race"] == sel["race"])
            cal_n = pd.Series(g["s_cal"]["SexxAge"].to_numpy()[yc == 1]).value_counts().to_dict()
            cal_ess = {c: kish(wc[(yc == 1) & (g["s_cal"]["SexxAge"].to_numpy() == c)]) for c in cal_n}
            auc_t = roc_auc_score(y["t_eval"], pr["t_eval"]) if 0 < y["t_eval"].mean() < 1 else np.nan
            common = dict(study=study, seed=seed, outcome=o, model=mname, model_backend=MODEL_BACKEND.get(mname), features=feat_tag,
                          heldout=unit, n_s_cal=len(yc), n_cal_pos=int(yc.sum()), auc_target=auc_t,
                          ess_frac_cal=kish(wc) / len(wc), ess_frac_cal_unclipped=kish(w["s_cal"][1]) / len(wc), clip_value=clipv, **gib_info)
            wdes = Ft["wdesign"].to_numpy()[P["t_eval"]][ok["t_eval"]]
            ev_lab = None if eval_by is None else Ft[eval_by].to_numpy()[P["t_eval"]][ok["t_eval"]]
            groups_eval = [("ALL", np.arange(len(y["t_eval"])))]
            if ev_lab is not None:
                groups_eval = [(u, ix) for u, ix in pd.Series(ev_lab).groupby(ev_lab).indices.items()] + groups_eval
            rr = np.random.default_rng(seed * 10007 + oi + unit_index * 31 + 101 * mi)
            for eu, ix in groups_eval:
                ye, ge = y["t_eval"][ix], g["t_eval"].iloc[ix].reset_index(drop=True)
                ref_m, ref_p = exact_reference(ye, ge, cal_n, ALPHA, CFG3["REF_SIMS"], rr)
                refe_m, refe_p = exact_reference(ye, ge, {c: math.floor(v) for c, v in cal_ess.items()}, ALPHA, CFG3["REF_SIMS"], rr)
                for meth, S in SETS.items():
                    Se = S[ix]
                    for wt in (["unweighted"] + (["design"] if not np.allclose(wdes, 1.0) else [])):
                        m = metrics(Se, ye, ge, w=None if wt == "unweighted" else wdes[ix])
                        rows.append(dict(**common, eval_unit=eu, method=meth, weighting=wt, **m,
                                         ref_exact_mean=ref_m, ref_exact_p05=ref_p, ref_ess_mean=refe_m, ref_ess_p05=refe_p,
                                         unit_seconds=time.time() - t0))
    tag = f"{study}__{str(unit).replace(' ', '_').replace('>', '')}__s{seed}"
    _save(rows, f"{UNIT_DIR}/{tag}.csv"); _save(glog, f"{UNIT_DIR}/{tag}.gibbs.csv")
    DONE.add(key); json.dump(sorted(map(list, DONE)), open(DONE_JSON, "w"))
    print(f"{study} {unit} seed {seed}: {len(rows)} rows, {time.time() - t0:.0f}s", flush=True)
    return "ok"


In [ ]:
# ======================= 5. Run the studies (resumable; each unit saved as soon as it finishes) =======================
def time_left(): return (time.time() - T0) / 60 < CFG3["TIME_BUDGET_MIN"]
DATA_INFO = {}
_prev_info = json.load(open(f"{OUT}/data_info.json")) if os.path.exists(f"{OUT}/data_info.json") else {}
def subset(X, num, cat, full=False):
    keep = list(X.columns) if full else [c for c in X.columns if c in CFG3["FEATURES_NO_PROXY"]]
    if not full:
        missing = [c for c in CFG3["FEATURES_NO_PROXY"] if c not in X.columns]
        if missing: print("no-proxy features not available in this file:", missing)
    return X[keep], [c for c in num if c in keep], [c for c in cat if c in keep]

def units_of(F):
    vc = F["state"].value_counts()
    return sorted(vc[vc >= CFG3["MIN_STATE_ROWS"]].index)

for study in CFG3["STUDIES"]:
    if not time_left(): print("time budget reached: re-run the notebook to continue"); break
    try:
        if study in ("R5", "LOSO", "LOSO_FULL"):
            if "kaggle" not in DATA_INFO: KF = load_kaggle(); DATA_INFO["kaggle"] = KF[4]
            F, X, num, cat, _ = KF
            X, num, cat = subset(X, num, cat, full=(study == "LOSO_FULL")); ftag = "full" if study == "LOSO_FULL" else "no_proxy"
            F = F.assign(region=F["state"].map(REGION))
            if study == "R5":
                units = [(r, np.where(F["region"] != r)[0], np.where(F["region"] == r)[0]) for r in REGIONS_ALL if (F["region"] == r).sum() > 500]
            else:   # LOSO and LOSO_FULL
                units = [(s, np.where(F["state"] != s)[0], np.where(F["state"] == s)[0]) for s in units_of(F)]
            for seed in CFG3["SEEDS"][study]:
                for ui, (u, s_, t_) in enumerate(units):
                    if not time_left(): break
                    run_unit(study, u, seed, F, X, s_, F, X, t_, num, cat, unit_index=ui, feat_tag=ftag)
        elif study == "RAW":
            if "raw2022" not in DATA_INFO: R22 = load_raw(2022); DATA_INFO["raw2022"] = R22[4]
            F, X, num, cat, _ = R22
            X, num, cat = subset(X, num, cat)
            units = [(s, np.where(F["state"] != s)[0], np.where(F["state"] == s)[0]) for s in units_of(F)]
            for seed in CFG3["SEEDS"][study]:
                for ui, (u, s_, t_) in enumerate(units):
                    if not time_left(): break
                    run_unit(study, u, seed, F, X, s_, F, X, t_, num, cat, unit_index=ui)
        elif study == "TEMP":
            if "raw2022" not in DATA_INFO: R22 = load_raw(2022); DATA_INFO["raw2022"] = R22[4]
            if "raw2023" not in DATA_INFO: R23 = load_raw(2023); DATA_INFO["raw2023"] = R23[4]
            common = [c for c in R22[1].columns if c in R23[1].columns and c in CFG3["FEATURES_NO_PROXY"]]   # no-proxy features asked in both years
            num = [c for c in R22[2] if c in common]; cat = [c for c in R22[3] if c in common]
            DATA_INFO["temporal_features"] = dict(used=common, dropped_2022_only=[c for c in R22[1].columns if c not in common])
            for seed in CFG3["SEEDS"][study]:
                if not time_left(): break
                run_unit("TEMP", "2022->2023", seed, R22[0], R22[1][common], np.arange(len(R22[0])), R23[0], R23[1][common],
                         np.arange(len(R23[0])), num, cat, eval_by="state")
    except Exception as e:
        traceback.print_exc()
        check(f"study {study} completed without error", False, repr(e))
if RAW_MAP_LOG: pd.DataFrame(RAW_MAP_LOG).drop_duplicates().to_csv(f"{OUT}/raw_variable_map.csv", index=False)
json.dump({**_prev_info, **DATA_INFO}, open(f"{OUT}/data_info.json", "w"), indent=2, default=str)
print("elapsed min", round((time.time() - T0) / 60, 1))


In [ ]:
# ======================= 6. Analysis: held-out units as the units of inference (pre-specified contrasts) =======================
# Unit = held-out state (LOSO, RAW), region (R5) or 2023 state (TEMP). Seeds are averaged within a unit first.
# For each contrast: mean and median paired difference, 95% percentile bootstrap CI over units (BOOT_STATES draws),
# Wilcoxon signed-rank and exact sign tests (two-sided), Holm adjustment within each (study, weighting) family.
# With 5 regions (R5) the smallest attainable two-sided sign-test p is 0.0625, as in the main paper; R5 is therefore
# descriptive and the state-level studies carry the inference. States are treated as exchangeable units; they are not
# a random sample, so the inference describes these states.
_uf = sorted(f for f in glob.glob(f"{UNIT_DIR}/*.csv") if not f.endswith(".gibbs.csv"))
U = pd.concat([pd.read_csv(f) for f in _uf], ignore_index=True) if _uf else pd.DataFrame()
if len(U): U.to_csv(UNITS_CSV, index=False)
_gf = sorted(glob.glob(f"{UNIT_DIR}/*.gibbs.csv"))
if _gf: pd.concat([pd.read_csv(f) for f in _gf], ignore_index=True).to_csv(GIBBS_LOG, index=False)
check("results file present", len(U) > 0, UNITS_CSV)
CONTRASTS = [  # (name, method A, method B, metric, better direction)
    ("GroupCond - ClassCond", "GroupCond", "ClassCond", "wg_cal_pos", "+"),
    ("GroupCond - ClassCond", "GroupCond", "ClassCond", "set_size", "-"),
    ("GroupCond - ClassCond", "GroupCond", "ClassCond", "wg_race_pos", "+"),
    ("SAGE-CP - GroupCond", "SAGE-CP", "GroupCond", "absdev_cov_pos", "-"),
    ("SAGE-CP - GroupCond", "SAGE-CP", "GroupCond", "wg_cal_pos", "+"),
    ("SAGE-CP - GroupCond", "SAGE-CP", "GroupCond", "set_size", "-"),
    ("Weighted+ClassCond - ClassCond", "Weighted+ClassCond", "ClassCond", "absdev_cov_pos", "-"),
    ("Gibbs-tuned - GroupCond", "Gibbs-tuned", "GroupCond", "wg_cal_pos", "+"),
    ("Gibbs-tuned - GroupCond", "Gibbs-tuned", "GroupCond", "absdev_cov_pos", "-"),
    ("Gibbs-tuned - GroupCond", "Gibbs-tuned", "GroupCond", "set_size", "-"),
    ("Gibbs-tuned - SAGE-CP", "Gibbs-tuned", "SAGE-CP", "wg_cal_pos", "+"),
    ("Gibbs-tuned - SAGE-CP", "Gibbs-tuned", "SAGE-CP", "set_size", "-"),
]
METRICS = ["coverage", "cov_pos", "absdev_cov_pos", "set_size", "both", "empty", "single1", "ppv_single1", "wg_cal_pos", "wg_race_pos",
           "auc_target", "ess_frac_cal", "ref_exact_mean", "ref_exact_p05", "ref_ess_mean", "ref_ess_p05", "n", "n_pos"]

def holm(p):
    p = np.asarray(p, float); o = np.argsort(np.where(np.isnan(p), np.inf, p)); m = np.sum(~np.isnan(p)); adj = np.full(len(p), np.nan); run = 0
    for r, i in enumerate(o):
        if np.isnan(p[i]): continue
        run = max(run, min(1.0, (m - r) * p[i])); adj[i] = run
    return adj

if len(U):
    U = U[~((U.study == "TEMP") & (U.eval_unit.astype(str) == "ALL"))]       # TEMP units are 2023 states; the national row is kept in units.csv
    U["unit"] = np.where(U.study == "TEMP", U.eval_unit.astype(str), U.heldout.astype(str))
    U.groupby(["study", "weighting", "outcome", "model", "method"])[METRICS].mean(numeric_only=True).to_csv(f"{OUT}/v3_summary_by_model.csv")
    # unit level: average over seeds AND base models within a unit (as the v2.1 region-level analysis)
    UL = U.groupby(["study", "weighting", "outcome", "unit", "method"], as_index=False)[METRICS].mean(numeric_only=True)
    # pooled over outcomes: mean of the three outcomes within a unit (only where all present)
    UP = UL.groupby(["study", "weighting", "unit", "method"], as_index=False)[METRICS].mean(numeric_only=True).assign(outcome="pooled")
    UL = pd.concat([UL, UP], ignore_index=True); UL.to_csv(f"{OUT}/v3_unit_level.csv", index=False)
    rb = np.random.default_rng(CFG3["BOOT_SEED"]); res = []
    for (st, wt, oc), d in UL.groupby(["study", "weighting", "outcome"]):
        for name, A, B, met, better in CONTRASTS:
            a = d[d.method == A].set_index("unit")[met]; b = d[d.method == B].set_index("unit")[met]
            diff = (a - b).dropna()
            if len(diff) == 0: continue
            n = len(diff); bs = rb.choice(diff.to_numpy(), size=(CFG3["BOOT_STATES"], n)).mean(1) if n > 1 else np.array([diff.mean()])
            nz = diff[diff != 0]
            p_w = wilcoxon(diff).pvalue if n >= 5 and len(nz) > 0 else np.nan
            p_s = binomtest(int((nz > 0).sum()), len(nz), 0.5).pvalue if len(nz) else np.nan
            res.append(dict(study=st, weighting=wt, outcome=oc, contrast=name, metric=met, better=better, n_units=n,
                            mean_diff=diff.mean(), median_diff=diff.median(), ci95_lo=np.quantile(bs, .025), ci95_hi=np.quantile(bs, .975),
                            n_A_better=int(((diff > 0) if better == "+" else (diff < 0)).sum()), n_ties=int((diff == 0).sum()),
                            p_wilcoxon=p_w, p_sign=p_s))
    C = pd.DataFrame(res)
    if len(C):
        C["p_wilcoxon_holm"] = np.nan
        for _, ix in C[C.outcome != "pooled"].groupby(["study", "weighting"]).groups.items():
            C.loc[ix, "p_wilcoxon_holm"] = holm(C.loc[ix, "p_wilcoxon"])
        for _, ix in C[C.outcome == "pooled"].groupby(["study", "weighting"]).groups.items():
            C.loc[ix, "p_wilcoxon_holm"] = holm(C.loc[ix, "p_wilcoxon"])
    C.to_csv(f"{OUT}/v3_contrasts.csv", index=False)
    # Proposition 3 reference: GroupCond is judged against the EXACT reference; SAGE-CP against the approximate ESS reference
    rg = []
    for (st, oc), d in UL[(UL.weighting == "unweighted") & (UL.outcome != "pooled")].groupby(["study", "outcome"]):
        for meth, rm, rp in [("GroupCond", "ref_exact_mean", "ref_exact_p05"), ("SAGE-CP", "ref_ess_mean", "ref_ess_p05"),
                             ("TargetOracle", "ref_exact_mean", "ref_exact_p05"), ("Gibbs-tuned", "ref_exact_mean", "ref_exact_p05")]:
            x = d[(d.method == meth)].dropna(subset=["wg_cal_pos", rm])
            if len(x): rg.append(dict(study=st, outcome=oc, method=meth, reference=rm.replace("_mean", ""), n_units=len(x),
                                      mean_gap=(x.wg_cal_pos - x[rm]).mean(), share_below_p05=(x.wg_cal_pos < x[rp]).mean()))
    pd.DataFrame(rg).to_csv(f"{OUT}/v3_reference_gaps.csv", index=False)
    SM = UL[UL.outcome != "pooled"].groupby(["study", "weighting", "outcome", "method"])[METRICS].mean(numeric_only=True)
    SM.to_csv(f"{OUT}/v3_summary_by_method.csv")
    with pd.option_context("display.width", 200, "display.max_columns", 30):
        print(C[C.outcome == "pooled"].round(4).to_string())
        print(pd.DataFrame(rg).round(4).to_string())
pd.DataFrame(CHECKS).to_csv(f"{OUT}/checks_v3.csv", index=False)
zp = f"{OUT}/../SAGE_CP_v3_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/**/*", recursive=True):
        if os.path.isfile(f) and not f.endswith(".zip"): z.write(f, os.path.relpath(f, OUT))
print("results zipped to", os.path.abspath(zp), "| elapsed min", round((time.time() - T0) / 60, 1))
